# Baseline experiments trên Colab T4

Notebook này điều phối ba baseline **ResNet-50 → DenseNet-121 → ViT-B/16** bằng `scripts/train.py`. Training logic vẫn nằm trong source code của repository.

## Trước khi chạy

1. Trong VS Code, chọn kernel **Colab** và cấp một runtime có GPU T4.
2. Đặt archive dữ liệu tại `MyDrive/DATH_CXR/datasets/consensus_v1_224.zip`.
3. Archive phải chứa folder gốc `consensus_v1_224/` với `images/`, `masks/`, `splits/`, `manifest.csv` và `metadata.json`.
4. Checkpoint và log sẽ được giữ trên Google Drive để có thể resume sau khi runtime bị ngắt.

> Notebook không commit hoặc push source code hay artifact lên GitHub.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/HuyDuongQuoc/DATH-HK261-explainable-cxr.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/DATH-HK261-explainable-cxr")

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "DATH_CXR"
DATA_ARCHIVE = DRIVE_ROOT / "datasets" / "consensus_v1_224.zip"
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
RUN_LOG_ROOT = DRIVE_ROOT / "run_logs"
EXPORT_PATH = Path("/content/exports/02_experiment_results.zip")

REQUIRE_T4 = True
EXPECTED_IMAGES = 15_000
EXPECTED_MASKS = 15_000
EXPECTED_TRAIN_ROWS = 10_500
EXPECTED_VAL_ROWS = 2_250
EXPECTED_TEST_ROWS = 2_250

EXPERIMENTS = [
    {
        "run_name": "resnet50_baseline_s701515_seed42_amp",
        "model_name": "resnet50",
        "config": "configs/resnet50_baseline.yaml",
    },
    {
        "run_name": "densenet121_baseline_s701515_seed42_amp",
        "model_name": "densenet121",
        "config": "configs/densenet121_baseline.yaml",
    },
    {
        "run_name": "vit_baseline_s701515_seed42_amp",
        "model_name": "vit_b_16",
        "config": "configs/vit_baseline.yaml",
    },
]

print("Experiments:", [item["run_name"] for item in EXPERIMENTS])
print("Data archive:", DATA_ARCHIVE)
print("Export path:", EXPORT_PATH)


Experiments: ['resnet50_baseline_s701515_seed42_amp', 'densenet121_baseline_s701515_seed42_amp', 'vit_baseline_s701515_seed42_amp']
Data archive: /content/drive/MyDrive/DATH_CXR/datasets/consensus_v1_224.zip
Export path: /content/exports/02_experiment_results.zip


## 1. Mount Google Drive

Drive là nơi lưu archive dữ liệu, checkpoint và run log bền vững.


In [2]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if not DATA_ARCHIVE.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy archive dữ liệu: {DATA_ARCHIVE}. "
        "Hãy upload consensus_v1_224.zip lên đúng vị trí rồi chạy lại cell."
    )

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_LOG_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive is ready:", DRIVE_ROOT)


Mounted at /content/drive
Drive is ready: /content/drive/MyDrive/DATH_CXR


## 2. Clone hoặc cập nhật source từ `main`

PAT chỉ được giữ trong memory của kernel. Để trống nếu repository có thể clone công khai. Notebook không ghi token vào file, output hoặc Git config.


In [3]:
import base64
import os
import subprocess
from getpass import getpass

github_token = getpass("GitHub PAT (để trống nếu repo public): " ).strip()

git_auth_env = os.environ.copy()
if github_token:
    auth = base64.b64encode(
        f"x-access-token:{github_token}".encode("utf-8")
    ).decode("ascii")
    git_auth_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.extraHeader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
    })

def run_git(arguments, cwd=None):
    command = ["git", *arguments]
    return subprocess.run(command, cwd=cwd, env=git_auth_env, check=True)

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(
            f"{REPO_DIR} đã tồn tại nhưng không phải Git repository. "
            "Hãy xóa folder này hoặc khởi tạo runtime mới."
        )
    run_git(["fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run_git(["checkout", REPO_BRANCH], cwd=REPO_DIR)
    run_git(["pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    run_git([
        "clone",
        "--branch", REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ])

github_token = None
auth = None
git_auth_env = None

commit_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Repository:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Commit:", commit_sha)


Repository: /content/DATH-HK261-explainable-cxr
Branch: main
Commit: ec68bfbd6b6dd991101a221a755b07241f8c467f


## 3. Cài dependencies và xác nhận T4

Không import PyTorch trước khi cài requirements để tránh giữ phiên bản cũ trong kernel.


In [4]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-common.txt")],
    check=True,
)
print("Dependencies installed.")


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-r', '/content/DATH-HK261-explainable-cxr/requirements-common.txt']' returned non-zero exit status 1.

In [ ]:
import torch
import torchvision

nvidia_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    check=False,
    capture_output=True,
    text=True,
)
if nvidia_smi.returncode != 0:
    raise RuntimeError(
        "Không gọi được nvidia-smi. Hãy tạo Colab server có GPU trong VS Code."
    )

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch không phát hiện CUDA. Không bắt đầu training trên CPU.")
if torch.version.cuda is None:
    raise RuntimeError("PyTorch hiện tại không được build với CUDA.")

gpu_name = torch.cuda.get_device_name(0)
if REQUIRE_T4 and "T4" not in gpu_name.upper():
    raise RuntimeError(
        f"Runtime đang dùng {gpu_name}, không phải Tesla T4. "
        "Hãy remove server và provision một Colab server T4."
    )

try:
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        _amp_probe = torch.ones(8, 8, device="cuda") @ torch.ones(8, 8, device="cuda")
    if not torch.isfinite(_amp_probe).all():
        raise RuntimeError("AMP probe produced non-finite values")
    del _amp_probe
except Exception as error:
    raise RuntimeError("Runtime T4 không hỗ trợ CUDA autocast FP16.") from error

print(nvidia_smi.stdout.strip())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("AMP FP16 probe: OK")


## 4. Giải nén và kiểm tra processed dataset

Dữ liệu được đọc từ disk local của Colab trong lúc train. Cell này idempotent: nếu dataset đã hợp lệ thì không giải nén lại.


In [ ]:
import shutil
import tempfile
import json
import zipfile

import pandas as pd

from src.data.splitting import assignment_sha256

PROCESSED_ROOT = REPO_DIR / "data" / "processed" / "consensus_v1_224"

def dataset_summary(root):
    required = [
        root / "images",
        root / "masks",
        root / "splits" / "train.csv",
        root / "splits" / "val.csv",
        root / "splits" / "test.csv",
        root / "splits" / "summary.csv",
        root / "splits" / "split_metadata.json",
        root / "manifest.csv",
        root / "metadata.json",
    ]
    if not all(path.exists() for path in required):
        return None

    tables = {
        name: pd.read_csv(root / "splits" / f"{name}.csv")
        for name in ("train", "val", "test")
    }
    id_sets = {name: set(table["image_id"].astype(str)) for name, table in tables.items()}
    if not id_sets["train"].isdisjoint(id_sets["val"]):
        raise ValueError("Train/validation overlap")
    if not id_sets["train"].isdisjoint(id_sets["test"]):
        raise ValueError("Train/test overlap")
    if not id_sets["val"].isdisjoint(id_sets["test"]):
        raise ValueError("Validation/test overlap")
    manifest_ids = set(pd.read_csv(root / "manifest.csv")["image_id"].astype(str))
    if set().union(*id_sets.values()) != manifest_ids:
        raise ValueError("Split IDs do not cover the processed manifest")
    split_metadata = json.loads(
        (root / "splits" / "split_metadata.json").read_text(encoding="utf-8")
    )
    calculated_assignment = assignment_sha256(tables)
    if split_metadata.get("assignment_sha256") != calculated_assignment:
        raise ValueError("Split assignment fingerprint mismatch")

    return {
        "images": sum(1 for _ in (root / "images").glob("*.png")),
        "masks": sum(1 for _ in (root / "masks").glob("*.npz")),
        "train_rows": len(pd.read_csv(root / "splits" / "train.csv")),
        "val_rows": len(pd.read_csv(root / "splits" / "val.csv")),
        "test_rows": len(pd.read_csv(root / "splits" / "test.csv")),
        "split_assignment_sha256": calculated_assignment,
    }

def validate_dataset(root):
    summary = dataset_summary(root)
    expected = {
        "images": EXPECTED_IMAGES,
        "masks": EXPECTED_MASKS,
        "train_rows": EXPECTED_TRAIN_ROWS,
        "val_rows": EXPECTED_VAL_ROWS,
        "test_rows": EXPECTED_TEST_ROWS,
    }
    actual_counts = {key: summary[key] for key in expected}
    if actual_counts != expected:
        raise ValueError(
            f"Processed dataset không đúng contract. Expected={expected}, actual={actual_counts}"
        )
    return summary

def safe_extract_zip(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            try:
                target.relative_to(destination)
            except ValueError as error:
                raise ValueError(f"Unsafe ZIP member: {member.filename}") from error
        archive.extractall(destination)

existing_summary = dataset_summary(PROCESSED_ROOT)
if existing_summary is not None:
    validate_dataset(PROCESSED_ROOT)
    print("Dataset đã tồn tại và hợp lệ; bỏ qua giải nén.")
else:
    if PROCESSED_ROOT.exists():
        raise RuntimeError(
            f"{PROCESSED_ROOT} đã tồn tại nhưng không đầy đủ. "
            "Hãy khởi tạo runtime mới để tránh ghi đè dữ liệu không rõ nguồn gốc."
        )

    temporary_root = Path(tempfile.mkdtemp(prefix="cxr-extract-", dir="/content"))
    try:
        safe_extract_zip(DATA_ARCHIVE, temporary_root)
        extracted_root = temporary_root / "consensus_v1_224"
        validate_dataset(extracted_root)
        PROCESSED_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(extracted_root), str(PROCESSED_ROOT))
    finally:
        shutil.rmtree(temporary_root, ignore_errors=True)

summary = validate_dataset(PROCESSED_ROOT)
print("Processed dataset ready:", PROCESSED_ROOT)
print(summary)


## 5. Training tuần tự và tự động resume

Mỗi epoch đều ghi `last.pt` lên Drive. Khi chạy lại notebook, experiment chưa hoàn tất sẽ resume; experiment đã đủ epoch hoặc đã early-stop sẽ được bỏ qua.


In [ ]:
import os

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.utils.config import load_config

def inspect_run(experiment):
    config_path = REPO_DIR / experiment["config"]
    config = load_config(config_path)
    checkpoint_dir = CHECKPOINT_ROOT / experiment["run_name"]
    result_dir = RUN_LOG_ROOT / experiment["run_name"]
    last_checkpoint = checkpoint_dir / "last.pt"
    best_checkpoint = checkpoint_dir / "best.pt"

    state = {
        "config_path": config_path,
        "config": config,
        "checkpoint_dir": checkpoint_dir,
        "result_dir": result_dir,
        "last_checkpoint": last_checkpoint,
        "resume": False,
        "complete": False,
    }

    if best_checkpoint.is_file() and not last_checkpoint.is_file():
        raise RuntimeError(
            f"Phát hiện best.pt nhưng thiếu last.pt trong {checkpoint_dir}. "
            "Không bắt đầu run mới để tránh ghi đè artifact cũ."
        )

    if last_checkpoint.is_file():
        checkpoint = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
        if checkpoint.get("model_name") != experiment["model_name"]:
            raise ValueError(f"Checkpoint backbone mismatch: {last_checkpoint}")
        if checkpoint.get("findings") != config["data"]["target_findings"]:
            raise ValueError(f"Checkpoint class order mismatch: {last_checkpoint}")
        if checkpoint.get("split_assignment_sha256") != summary["split_assignment_sha256"]:
            raise ValueError(f"Checkpoint split fingerprint mismatch: {last_checkpoint}")
        if checkpoint.get("amp_enabled") is not True:
            raise ValueError(f"Checkpoint không thuộc AMP protocol: {last_checkpoint}")

        completed_epochs = int(checkpoint["epoch"]) + 1
        reached_limit = completed_epochs >= int(config["training"]["epochs"])
        early_stopped = int(checkpoint["bad_epochs"]) >= int(
            config["training"]["patience"]
        )
        state["complete"] = reached_limit or early_stopped
        state["resume"] = not state["complete"]
        state["completed_epochs"] = completed_epochs

    return state

def train_experiment(experiment):
    state = inspect_run(experiment)
    run_name = experiment["run_name"]
    state["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
    state["result_dir"].mkdir(parents=True, exist_ok=True)

    if state["complete"]:
        print(f"[{run_name}] đã hoàn tất; bỏ qua training.")
    else:
        command = [
            sys.executable,
            str(REPO_DIR / "scripts" / "train.py"),
            "--config", str(state["config_path"]),
            "--checkpoint-dir", str(state["checkpoint_dir"]),
            "--result-dir", str(state["result_dir"]),
        ]
        if state["resume"]:
            command.extend(["--resume", str(state["last_checkpoint"])])
            print(f"[{run_name}] resume từ {state['last_checkpoint']}")
        else:
            print(f"[{run_name}] bắt đầu run mới")

        training_env = os.environ.copy()
        current_pythonpath = training_env.get("PYTHONPATH")
        training_env["PYTHONPATH"] = (
            str(REPO_DIR)
            if not current_pythonpath
            else str(REPO_DIR) + os.pathsep + current_pythonpath
        )
        subprocess.run(
            command,
            cwd=REPO_DIR,
            env=training_env,
            check=True,
        )

    best_checkpoint = state["checkpoint_dir"] / "best.pt"
    if not best_checkpoint.is_file():
        raise FileNotFoundError(f"Không tìm thấy best checkpoint: {best_checkpoint}")
    return inspect_run(experiment)


In [ ]:
import json

def load_best_metrics(experiment):
    state = inspect_run(experiment)
    best_checkpoint_path = state["checkpoint_dir"] / "best.pt"
    metrics_path = state["result_dir"] / "best_val_metrics.json"
    if not best_checkpoint_path.is_file():
        raise FileNotFoundError(best_checkpoint_path)
    if not metrics_path.is_file():
        raise FileNotFoundError(metrics_path)

    checkpoint = torch.load(best_checkpoint_path, map_location="cpu", weights_only=False)
    findings = state["config"]["data"]["target_findings"]
    if checkpoint.get("model_name") != experiment["model_name"]:
        raise ValueError(f"Best checkpoint backbone mismatch: {best_checkpoint_path}")
    if checkpoint.get("findings") != findings:
        raise ValueError(f"Best checkpoint class order mismatch: {best_checkpoint_path}")
    if checkpoint.get("split_assignment_sha256") != summary["split_assignment_sha256"]:
        raise ValueError(f"Best checkpoint split fingerprint mismatch: {best_checkpoint_path}")
    if checkpoint.get("amp_enabled") is not True:
        raise ValueError(f"Best checkpoint không thuộc AMP protocol: {best_checkpoint_path}")

    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    if metrics.get("amp_enabled") is not True:
        raise ValueError(f"Best metrics không thuộc AMP protocol: {metrics_path}")
    if metrics.get("split_assignment_sha256") != summary["split_assignment_sha256"]:
        raise ValueError(f"Best metrics split fingerprint mismatch: {metrics_path}")
    print(
        f"[{experiment['run_name']}] best epoch={metrics['checkpoint_epoch']}, "
        f"macro-AUROC={metrics['macro_auroc']:.4f}"
    )
    return metrics

best_metrics_by_run = {}
for experiment_index, experiment in enumerate(EXPERIMENTS, start=1):
    print(f"[{experiment_index}/{len(EXPERIMENTS)}] {experiment['run_name']}")
    train_experiment(experiment)
    best_metrics_by_run[experiment["run_name"]] = load_best_metrics(experiment)

print("All experiments completed and evaluated.")


## 6. Tổng hợp và đóng gói kết quả

ZIP chỉ chứa metrics, config và metadata tái lập. Checkpoint `.pt` không được đưa vào gói tải về.


In [ ]:
import hashlib
from datetime import datetime, timezone

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

summary_rows = []
for experiment in EXPERIMENTS:
    run_name = experiment["run_name"]
    result_dir = RUN_LOG_ROOT / run_name
    history_path = result_dir / "history.csv"
    metrics_path = result_dir / "best_val_metrics.json"
    if not history_path.is_file() or not metrics_path.is_file():
        raise FileNotFoundError(f"Thiếu artifact để export cho {run_name}")

    history = pd.read_csv(history_path)
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    summary_rows.append({
        "experiment": run_name,
        "model": experiment["model_name"],
        "best_epoch": int(metrics["checkpoint_epoch"]),
        "epochs_completed": int(len(history)),
        "val_macro_auroc": float(metrics["macro_auroc"]),
        "val_macro_average_precision": float(metrics["macro_average_precision"]),
        "val_macro_f1": float(metrics["macro_f1"]),
        "total_train_seconds": float(history["train_seconds"].sum()),
        "total_val_seconds": float(history["val_seconds"].sum()),
        "mean_train_images_per_second": float(history["train_images_per_second"].mean()),
        "mean_val_images_per_second": float(history["val_images_per_second"].mean()),
        "peak_gpu_memory_gb": float(history["peak_gpu_memory_gb"].max()),
    })

source_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()
default_config = load_config(REPO_DIR / "configs" / "default.yaml")
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository_url": REPO_URL,
    "repository_branch": REPO_BRANCH,
    "git_commit": commit_sha,
    "git_worktree_status": source_status,
    "gpu": gpu_name,
    "pytorch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "cuda_version": torch.version.cuda,
    "amp_enabled": bool(default_config["training"]["use_amp"]),
    "data_loader": {
        "batch_size": int(default_config["training"]["batch_size"]),
        "num_workers": int(default_config["training"]["num_workers"]),
        "persistent_workers": bool(default_config["training"]["persistent_workers"]),
        "prefetch_factor": int(default_config["training"]["prefetch_factor"]),
    },
    "seed": int(default_config["seed"]),
    "data_archive": str(DATA_ARCHIVE),
    "data_archive_sha256": sha256_file(DATA_ARCHIVE),
    "processed_dataset": summary,
    "split_assignment_sha256": summary["split_assignment_sha256"],
    "experiments": [item["run_name"] for item in EXPERIMENTS],
    "checkpoint_root": str(CHECKPOINT_ROOT),
    "checkpoint_included_in_export": False,
}

staging_root = Path(tempfile.mkdtemp(prefix="cxr-export-", dir="/content"))
metrics_root = staging_root / "outputs" / "metrics"
metrics_root.mkdir(parents=True, exist_ok=True)

required_files = [
    "config.json",
    "history.csv",
    "latest_val_metrics.json",
    "best_val_metrics.json",
]
for experiment in EXPERIMENTS:
    run_name = experiment["run_name"]
    source_dir = RUN_LOG_ROOT / run_name
    destination_dir = metrics_root / run_name
    destination_dir.mkdir(parents=True, exist_ok=True)
    for filename in required_files:
        source_file = source_dir / filename
        if not source_file.is_file():
            raise FileNotFoundError(f"Thiếu artifact: {source_file}")
        shutil.copy2(source_file, destination_dir / filename)

pd.DataFrame(summary_rows).to_csv(metrics_root / "baseline_summary.csv", index=False)
(metrics_root / "experiment_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
if EXPORT_PATH.exists():
    EXPORT_PATH.unlink()
shutil.make_archive(
    str(EXPORT_PATH.with_suffix("")),
    "zip",
    root_dir=staging_root,
)

with zipfile.ZipFile(EXPORT_PATH) as archive:
    members = sorted(name for name in archive.namelist() if not name.endswith("/"))
if any(name.lower().endswith((".pt", ".pth", ".ckpt")) for name in members):
    raise RuntimeError("Export ZIP chứa checkpoint ngoài ý muốn")
for experiment in EXPERIMENTS:
    prefix = f"outputs/metrics/{experiment['run_name']}/"
    if not all(prefix + filename in members for filename in required_files):
        raise RuntimeError(f"Export ZIP thiếu artifact cho {experiment['run_name']}")

shutil.rmtree(staging_root, ignore_errors=True)
print(pd.DataFrame(summary_rows).to_string(index=False))
print("Export ready:", EXPORT_PATH)
print("Files in ZIP:", len(members))


## 7. Tải kết quả về repository trên laptop

Trong VS Code:

1. Mở biểu tượng **Colab** ở Activity Bar.
2. Trong `Contents`, tìm `/content/exports/02_experiment_results.zip`.
3. Right-click file → **Download...**.
4. Lưu thành `D:\Junior\ĐATH\main\outputs\02_experiment_results.zip`.
5. Từ repo root trên laptop, giải nén bằng PowerShell:

```powershell
Expand-Archive -LiteralPath .\outputs\02_experiment_results.zip -DestinationPath . -Force
```

Kết quả sẽ nằm trong `outputs/metrics/`. Checkpoint vẫn ở `MyDrive/DATH_CXR/checkpoints/` để phục vụ resume và các phase Grad-CAM sau này.

Notebook không tự động commit hoặc push bất kỳ file nào.
